> **Review note (2026-10-21):** The submitted workflow had data-leakage and model-selection risks. This portfolio draft fixes train/test scaling leakage and excludes Final_Grade pending prediction-time justification. For a more rigorous baseline, use src/evaluate_models.py, which fits preprocessing within CV folds, selects using training-only CV, and evaluates on a held-out test set. The dataset is not included pending license/provenance review. Historical scores are not independently validated and must not be presented as verified results.

# CS465 – Student Dropout Prediction
## Phase 3: Dataset Characterization, Feature Engineering & Methodology

**Team members:** Areen Al Ruwaitea and Layal Alsultan.


In [ ]:
# import all libraries we need
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.over_sampling import SMOTE

import xgboost as xgb

print('all libraries imported successfully')

---
## Section 1 – Dataset Characterization, Preprocessing & EDA

In [ ]:
data_path = os.environ.get('DROPOUT_DATA_PATH', 'data/raw/student dropout.csv')
df = pd.read_csv(data_path)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found')

In [ ]:
# basic statistical summary of numerical features
df.describe().round(2)

In [ ]:
# check the class distribution of the target variable
df['Dropped_Out'] = df['Dropped_Out'].astype(int)

counts = df['Dropped_Out'].value_counts().sort_index()
pcts   = df['Dropped_Out'].value_counts(normalize=True).sort_index() * 100 # % of each class in dataset
print('Class Distribution:')
print(f'  Not Dropped Out (0): {counts[0]}  ({pcts[0]:.1f}%)')
print(f'  Dropped Out     (1): {counts[1]}  ({pcts[1]:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Not Dropped Out', 'Dropped Out'], [counts[0], counts[1]],
       color=['steelblue', 'tomato'], edgecolor='black', width=0.5)

for i, (v, p) in enumerate(zip([counts[0], counts[1]], [pcts[0], pcts[1]])):
    ax.text(i, v + 5, f'{v}\n({p:.1f}%)', ha='center', fontsize=11) # to label bars

ax.set_title('Class Distribution - Dropped Out vs Not', fontsize=13)
ax.set_ylabel('Number of Students')
ax.set_ylim(0, max(counts) * 1.2) # boundary of y-axis from 0-20% taller than max bar
plt.tight_layout()
plt.show()


In [ ]:
# encode all categorical columns (obj type) with label encoding
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', cat_cols)

le = LabelEncoder() # text to no.
df_encoded = df.copy() #copy of encoded df
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df[col])

print('Encoding done. Sample:')
df_encoded.head(3)

In [ ]:
# plot distribution of a few key numerical features by class
num_features = ['Age', 'Study_Time', 'Number_of_Failures', 'Number_of_Absences',
                'Grade_1', 'Grade_2', 'Final_Grade']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(num_features):
    for label, color, name in [(0, 'steelblue', 'Not Dropped'), (1, 'tomato', 'Dropped')]:
        axes[i].hist(df[df['Dropped_Out'] == label][feat].dropna(),
                     alpha=0.6, color=color, label=name, bins=15, edgecolor='white')
    axes[i].set_title(feat, fontsize=11)
    axes[i].legend(fontsize=8)


fig.suptitle('Feature Distributions by Dropout Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap to see which features relate to each other
plt.figure(figsize=(16, 12))
corr = df_encoded.corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=False, linewidths=0.4, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# top correlations with the target variable specifically
target_corr = df_encoded.corr()['Dropped_Out'].drop('Dropped_Out').sort_values(key=abs, ascending=False)
print('Top 10 features correlated with Dropped_Out:')
print(target_corr.head(10).round(3))

fig, ax = plt.subplots(figsize=(8, 5))
top10 = target_corr.head(10)
colors = ['tomato' if v > 0 else 'steelblue' for v in top10.values]
ax.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1], edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 10 Feature Correlations with Dropout', fontsize=13)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

# Red +ve correlation
# Blue -ve correlation

In [ ]:
# box plots for grade features by dropout class
grade_cols = ['Grade_1', 'Grade_2', 'Final_Grade']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, col in enumerate(grade_cols):
    data_0 = df[df['Dropped_Out'] == 0][col]
    data_1 = df[df['Dropped_Out'] == 1][col]
    axes[i].boxplot([data_0, data_1], labels=['Not Dropped', 'Dropped'],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontsize=12)
    axes[i].set_ylabel('Grade')

fig.suptitle('Grade Distribution by Dropout Status', fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 2 – Feature Engineering & Baseline Reproduction

In [ ]:
# create a few engineered features based on EDA findings
df_feat = df_encoded.copy()

# Conservative portfolio baseline: omit Final_Grade pending prediction-time justification.
df_feat['Avg_Grade'] = (df_feat['Grade_1'] + df_feat['Grade_2']) / 2

# grade improvement from grade 1 to grade 2
df_feat['Grade_Improvement'] = df_feat['Grade_2'] - df_feat['Grade_1']

# flag for students with any failures (common dropout risk indicator)
df_feat['Has_Failures'] = (df_feat['Number_of_Failures'] > 0).astype(int)

# high absence flag (above median)
median_abs = df_feat['Number_of_Absences'].median()
df_feat['High_Absences'] = (df_feat['Number_of_Absences'] > median_abs).astype(int)

print('New features added: Avg_Grade, Grade_Improvement, Has_Failures, High_Absences')
df_feat[['Avg_Grade', 'Grade_Improvement', 'Has_Failures', 'High_Absences', 'Dropped_Out']].head()

In [ ]:
# Separate features and target. Final_Grade is excluded above pending prediction-time justification.
X = df_feat.drop(columns=['Dropped_Out', 'Final_Grade'], errors='ignore')
y = df_feat['Dropped_Out']
print('Feature matrix shape:', X.shape)
print('Target distribution:', y.value_counts().to_dict())


In [ ]:
# Stratified split first; fit scaling on training data only to avoid test-set leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print('Training set:', X_train.shape, '| Dropout rate:', round(y_train.mean()*100, 1), '%')
print('Test set:    ', X_test.shape,  '| Dropout rate:', round(y_test.mean()*100, 1), '%')


In [ ]:
# apply SMOTE on training data only to handle class imbalance
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print('After SMOTE:')
print('  Training set size:', X_train_sm.shape)
print('  Class distribution:', pd.Series(y_train_sm).value_counts().to_dict())

---
## Section 3 – Methodology, Benchmarking & Experimental Evaluation

In [ ]:
# define all models we want to compare (baselines + stronger models)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(probability=True, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=100, random_state=42,
                                              use_label_encoder=False, eval_metric='logloss')
}
print('Models defined:', list(models.keys()))

In [ ]:
# helper function to compute all our benchmark metrics for a model
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred)
    rec  = recall_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    spec  = tn / (tn + fp)
    gmean = np.sqrt(rec * spec)

    return {'Model': name, 'Accuracy': round(acc,4), 'F1-Score': round(f1,4),
            'Recall': round(rec,4), 'Precision': round(prec,4),
            'G-Mean': round(gmean,4)}

print('evaluation function ready')

In [ ]:
# run all models WITHOUT SMOTE first (baseline)
results_no_smote = []
for name, model in models.items():
    r = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results_no_smote.append(r)
    print(f'{name}: F1={r["F1-Score"]}, Recall={r["Recall"]}')

df_results_base = pd.DataFrame(results_no_smote).sort_values('F1-Score', ascending=False)
print('\nBaseline results (no SMOTE):')
df_results_base

In [ ]:
# run all models WITH SMOTE
results_smote = []
for name, model in models.items():
    r = evaluate_model(name, model, X_train_sm, y_train_sm, X_test, y_test)
    results_smote.append(r)
    print(f'{name}: F1={r["F1-Score"]}, Recall={r["Recall"]}')

df_results_smote = pd.DataFrame(results_smote).sort_values('F1-Score', ascending=False)
print('\nResults with SMOTE:')
df_results_smote

In [ ]:
# bar chart comparing F1 scores across all models (with vs without SMOTE)
model_names = df_results_base['Model'].tolist()
f1_base  = df_results_base.set_index('Model').loc[model_names, 'F1-Score'].values
f1_smote = df_results_smote.set_index('Model').loc[model_names, 'F1-Score'].values

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width/2, f1_base,  width, label='Without SMOTE', color='steelblue', edgecolor='black')
ax.bar(x + width/2, f1_smote, width, label='With SMOTE',    color='tomato',    edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.1)
ax.set_title('F1-Score Comparison: All Models (With vs Without SMOTE)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# confusion matrix for best model (by F1) with SMOTE
best_model_name = df_results_smote.iloc[0]['Model']
best_model = models[best_model_name]
best_model.fit(X_train_sm, y_train_sm)
y_pred_best = best_model.predict(X_test)

print(f'Best model: {best_model_name}')
print(classification_report(y_test, y_pred_best, target_names=['Not Dropped', 'Dropped']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['Not Dropped', 'Dropped'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix – {best_model_name}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# feature importance from Random Forest (good interpretability)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sm, y_train_sm)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top15 = importances.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15.index[::-1], top15.values[::-1], color='steelblue', edgecolor='black')
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Feature Importances – Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

print(top15.round(4))

In [ ]:
# For leakage-aware cross-validation and model selection, use src/evaluate_models.py.
# That script fits preprocessing inside each CV fold and reserves a held-out test set
# for the final evaluation. It uses class weighting rather than SMOTE.
# Place the authorized dataset at data/raw/student dropout.csv, then run:
# python src/evaluate_models.py
print('Use src/evaluate_models.py for the separated evaluation path; no project metrics are reproduced here.')


In [ ]:
# final summary table of all results with SMOTE
print('=== FINAL RESULTS SUMMARY (With SMOTE) ===')
print(df_results_smote.to_string(index=False))